# Download and extract `layer_out/27` activations

Stripped-down sibling of `layer_pls_scores_block_normalized.ipynb`: the download and staging
half only - no PLS, no residual PCA, no scoring.

For each dataset it streams the cached `.pt` batch files out of GCS, pulls the `layer_out/27`
rows at the configured prompt position, and writes them into a memory-mapped `.npy` array
alongside a metadata CSV with one row per activation row. The `.pt` files are deleted as they
are consumed, so peak disk use stays at `DOWNLOAD_WORKERS` batch files rather than a whole
folder.

Rows whose prompt declares no time horizon are kept here (`REQUIRE_HORIZON = False`), since
nothing in this notebook needs a target; their horizon columns are `NaN`. Set it to `True` to
reproduce the filtering the PLS notebook does.

## 1. Setup

Clones the repo, installs what Colab does not ship, and puts `.env` in place. Skip this cell
when running locally against a checkout that already has a `.env`.

In [ ]:
REPO_URL = 'https://github.com/SD-interp/test-temporal.git'
REPO_DIR = 'test-temporal'

import os
from pathlib import Path

if not Path(REPO_DIR).exists() and Path.cwd().name != REPO_DIR:
    !git clone {REPO_URL} {REPO_DIR}
if Path(REPO_DIR).exists():
    os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())

# Colab ships numpy/pandas/torch; these two are the usual gaps.
!pip install -q google-cloud-storage python-dotenv

# The repo tracks `.env.example`, not `.env` (which would carry credentials). `-n` keeps an
# existing `.env` if this cell is re-run.
!mv -n .env.example .env
# Printed rather than `cat`-ed, so a filled-in credential never lands in the output.
print('.env present:', Path('.env').exists())

In [ ]:
import gc
import json
import os
import shutil
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from google.cloud import storage
from numpy.lib.format import open_memmap
from tqdm.auto import tqdm

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
load_dotenv(repo_root / '.env')

## 2. Authenticate to Google Cloud

`gcloud auth application-default login` installs Application Default Credentials, which the
storage client picks up on its own. In Colab it prints a URL to open and a code to paste back;
locally it opens a browser. Without it the client falls back to the Compute Engine metadata
service and fails with a `RefreshError`.

In [ ]:
!gcloud auth application-default login

PROJECT_ID = os.getenv('GCP_PROJECT_ID') or 'temporal-interp-exp'
!gcloud config set project {PROJECT_ID}
print(f'Project: {PROJECT_ID}')

## 3. Configure

Layer 27 lives in the `expanded_` caches (layers 17-35), so `RANGE_TAG` points there.

**On cost and disk:** each cached batch file holds all nineteen layers at both positions
(~25 MB) even though only layer 27 at one position is kept, so extracting a dataset in full
means downloading its whole folder - `conversational` alone is 2,618 files, ~65 GB of transfer.
That 65 GB never sits on disk at once: batch files are deleted as they are read, so the download
directory holds at most `DOWNLOAD_WORKERS` of them (~200 MB). What persists is the extracted
`.npy`, at `rows * 2560 * 4` bytes - roughly 3.4 GB for `conversational` and well under 10 GB for
all five datasets together.

`estimate_disk()` below prints that projection against the free space actually available before
anything downloads, so a run that would not fit is caught up front rather than 60 GB in. Set
`BATCH_SAMPLE_SIZE` to take a spread-out subset of files instead of everything, and
`MAX_ROWS_PER_BATCH` to thin each file.

In [ ]:
BUCKET_NAME = 'temporal-research-bucket'
RANGE_TAG = 'expanded_'                 # Folder tag holding layers 17-35.
LAYER = 27                              # The layer to extract.
POSITION: int | list[int] = -1          # Prompt token(s) to extract; a list concatenates them.


@dataclass(frozen=True)
class Dataset:
    name: str
    gcs_suffix: str


DATASETS = [
    Dataset('conversational_no_output_format', 'NOF_selected_acts'),
    Dataset('abstract', 'abstract_selected_acts'),
    Dataset('plain_english', 'plain_english_selected_acts'),
    Dataset('plain_long', 'plain_long_selected_acts'),
    Dataset('conversational', 'selected_acts'),     # 2,618 files, ~65 GB to stream through.
]

DATA_DIR = repo_root / 'data' / f'layer{LAYER}_activations'

BATCH_SAMPLE_SIZE: int | None = None    # Batch files to take, evenly spaced; None takes every file.
MAX_ROWS_PER_BATCH: int | None = None   # Rows kept per file; None keeps all 128.
DOWNLOAD_WORKERS = 8
REQUIRE_HORIZON = False                 # Drop rows whose prompt declares no time horizon.
INCLUDE_PROMPTS = False                 # Write the full prompt text into the metadata CSV.
DELETE_BATCHES = True                   # Remove each downloaded `.pt` as soon as it is read.
RANDOM_SEED = 0

LAYER_COMPONENT = f'layer_out/{LAYER}'
POSITIONS = [POSITION] if isinstance(POSITION, int) else list(POSITION)
if not POSITIONS:
    raise ValueError('POSITION must name at least one prompt token.')
if len(set(POSITIONS)) != len(POSITIONS):
    raise ValueError(f'POSITION repeats a token: {POSITIONS}')
POSITION_TAG = '_'.join(str(position) for position in POSITIONS)

rng = np.random.default_rng(RANDOM_SEED)
DATA_DIR.mkdir(parents=True, exist_ok=True)
client = storage.Client(project=PROJECT_ID)
position_label = (
    f'position {POSITIONS[0]}' if len(POSITIONS) == 1
    else f'positions {POSITIONS} concatenated'
)
print(f'Extracting {LAYER_COMPONENT} at {position_label} for {len(DATASETS)} dataset(s).')
print(f'Writing to {DATA_DIR}')

## 4. Helpers

Horizon conversion matches `layer_pls_scores_block_normalized.ipynb` and
`scripts/fit_expanded_pls_residual_pca.py`, so `log10_time_horizon_months` means the same thing
everywhere.

In [ ]:
UNIT_TO_MONTHS = {
    'second': 1 / (30.4375 * 86400), 'minute': 1 / (30.4375 * 1440),
    'hour': 1 / (30.4375 * 24), 'day': 1 / 30.4375, 'week': 7 / 30.4375,
    'month': 1.0, 'year': 12.0, 'decade': 120.0, 'century': 1200.0, 'millennium': 12000.0,
}
UNIT_TO_MONTHS.update({f'{unit}s': value for unit, value in list(UNIT_TO_MONTHS.items())})
UNIT_TO_MONTHS['centuries'] = 1200.0
UNIT_TO_MONTHS['millennia'] = 12000.0


def horizon_months(metadata):
    """Return a prompt's horizon in months, or None when it declares none."""
    value = metadata.get('base_value', metadata.get('value'))
    unit = metadata.get('base_unit', metadata.get('unit'))
    if value in (None, 'N/A') or unit in (None, 'N/A'):
        return None
    unit_key = str(unit).lower()
    if unit_key not in UNIT_TO_MONTHS:
        raise ValueError(f'Cannot convert time-horizon unit {unit!r} to months.')
    months = float(value) * UNIT_TO_MONTHS[unit_key]
    return months if months > 0 else None


def flatten_metadata(metadata):
    """Flatten one prompt's nested metadata into dotted scalar fields."""
    flattened = {}
    for key, value in metadata.items():
        if isinstance(value, dict):
            flattened.update({f'{key}.{inner}': item for inner, item in value.items()})
        elif not isinstance(value, (list, tuple, set)):
            flattened[key] = value
    return flattened


def folder_prefix(dataset):
    return f'{RANGE_TAG}{dataset.gcs_suffix}'


def list_batch_blobs(dataset):
    """Every activation batch blob in one dataset folder, ordered by name."""
    prefix = folder_prefix(dataset)
    blobs = sorted(
        (
            blob for blob in client.bucket(BUCKET_NAME).list_blobs(prefix=prefix + '/')
            if Path(blob.name).name.startswith('activations_batch_') and blob.name.endswith('.pt')
        ),
        key=lambda blob: blob.name,
    )
    if not blobs:
        raise FileNotFoundError(f'No activation batches below gs://{BUCKET_NAME}/{prefix}')
    if BATCH_SAMPLE_SIZE is None or len(blobs) <= BATCH_SAMPLE_SIZE:
        return blobs
    picks = np.linspace(0, len(blobs) - 1, BATCH_SAMPLE_SIZE).round().astype(int)
    sampled = [blobs[position] for position in dict.fromkeys(picks.tolist())]
    print(f'  sampling {len(sampled)} of {len(blobs)} batch files, evenly spaced.')
    return sampled


def selected_features(payload):
    """Return this batch's `layer_out/LAYER` rows at every configured position.

    One position gives the plain (rows, hidden) block; several are laid end to end along the
    feature axis in `POSITIONS` order, giving (rows, hidden * len(POSITIONS)).
    """
    activations = payload['activations']
    if LAYER_COMPONENT not in activations:
        available = sorted(activations)
        raise KeyError(
            f'{LAYER_COMPONENT} not in this batch; it holds {available[0]}..{available[-1]}.'
        )
    cached = list(payload['positions'])
    missing_positions = [position for position in POSITIONS if position not in cached]
    if missing_positions:
        raise ValueError(f'This batch cached positions {cached}; it is missing {missing_positions}.')
    blocks = [
        activations[LAYER_COMPONENT][:, cached.index(position), :].to(torch.float32)
        for position in POSITIONS
    ]
    return blocks[0] if len(blocks) == 1 else torch.cat(blocks, dim=1)

Before downloading anything, check that the extracted arrays will fit. The estimate counts one
row per prompt in every batch file (128 rows per file, `HIDDEN_SIZE` floats each) plus the
transient batch files held during a download group.

In [ ]:
HIDDEN_SIZE = 2560          # Per position; the array width is HIDDEN_SIZE * len(POSITIONS).
ROWS_PER_BATCH_FILE = 128
BATCH_FILE_BYTES = 25e6


def estimate_disk():
    """Project persistent and peak disk use, and compare it with the free space here."""
    row_bytes = HIDDEN_SIZE * len(POSITIONS) * 4
    total = 0.0
    for dataset in DATASETS:
        file_count = len(list_batch_blobs(dataset))
        rows = file_count * (MAX_ROWS_PER_BATCH or ROWS_PER_BATCH_FILE)
        array_gb = rows * row_bytes / 1e9
        total += array_gb
        print(f'{dataset.name}: {file_count:,} files (~{file_count * BATCH_FILE_BYTES / 1e9:.0f} GB '
              f'to stream) -> ~{rows:,} rows, ~{array_gb:.2f} GB kept')
    transient_gb = DOWNLOAD_WORKERS * BATCH_FILE_BYTES / 1e9
    free_gb = shutil.disk_usage(DATA_DIR).free / 1e9
    print(f'\nPersistent: ~{total:.2f} GB of arrays; peak adds ~{transient_gb:.2f} GB of batch files.')
    print(f'Free on this volume: {free_gb:.1f} GB')
    if total + transient_gb > free_gb * 0.9:
        print('WARNING: that is within 10% of the free space. Lower BATCH_SAMPLE_SIZE or '
              'MAX_ROWS_PER_BATCH, or run the datasets one at a time.')
    return total


_ = estimate_disk()

## 5. Stage one dataset

Batch files are fetched `DOWNLOAD_WORKERS` at a time, the layer 27 rows are copied into a
memory-mapped `.npy` array, and each `.pt` file is deleted as soon as it has been read. The
array is sized from an upper bound on rows per file, so it is rewritten without its unused tail
once the folder is exhausted.

In [ ]:
def trim_to_rows(features_path, row_count):
    """Rewrite a staged array without the unused tail rows left by the capacity estimate."""
    array = np.load(features_path, mmap_mode='r')
    if array.shape[0] == row_count:
        del array
        return
    trimmed_path = features_path.with_name(features_path.stem + '.trimmed.npy')
    trimmed = open_memmap(trimmed_path, mode='w+', dtype=array.dtype,
                          shape=(row_count, array.shape[1]))
    for start in range(0, row_count, 8_192):
        # `array` still has the full capacity, so the stop has to be clamped to `row_count`;
        # slicing it by chunk size alone hands back more rows than the last chunk can hold.
        stop = min(start + 8_192, row_count)
        trimmed[start:stop] = array[start:stop]
    trimmed.flush()
    del trimmed, array
    gc.collect()
    trimmed_path.replace(features_path)


def stage_dataset(dataset):
    """Download one dataset and return (memmap path, row count, metadata frame)."""
    blobs = list_batch_blobs(dataset)
    download_dir = DATA_DIR / 'batches' / dataset.name
    download_dir.mkdir(parents=True, exist_ok=True)
    features_path = DATA_DIR / f'{dataset.name}_layer{LAYER}_pos{POSITION_TAG}.npy'

    array = None
    hidden_size = 0
    capacity = 0
    write_cursor = 0
    records = []
    dropped = 0

    def fetch(blob):
        local_path = download_dir / Path(blob.name).name
        blob.download_to_filename(str(local_path))
        return local_path

    progress = tqdm(total=len(blobs), desc=f'{dataset.name}: staging', leave=False)
    executor = ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS)
    try:
        for group_start in range(0, len(blobs), DOWNLOAD_WORKERS):
            group = blobs[group_start:group_start + DOWNLOAD_WORKERS]
            for local_path in list(executor.map(fetch, group)):
                try:
                    payload = torch.load(local_path, map_location='cpu', weights_only=True, mmap=True)
                    features = selected_features(payload)
                    metadata_rows = payload['prompt_metadata']
                    if len(metadata_rows) != features.shape[0]:
                        raise ValueError(f'{local_path.name}: metadata and activations disagree on rows.')

                    keep, keep_horizons = [], []
                    for offset, metadata in enumerate(metadata_rows):
                        months = horizon_months(metadata)
                        if months is None and REQUIRE_HORIZON:
                            dropped += 1
                            continue
                        keep.append(offset)
                        keep_horizons.append(months)
                    if MAX_ROWS_PER_BATCH is not None and len(keep) > MAX_ROWS_PER_BATCH:
                        chosen = np.sort(rng.choice(len(keep), size=MAX_ROWS_PER_BATCH, replace=False))
                        keep = [keep[position] for position in chosen]
                        keep_horizons = [keep_horizons[position] for position in chosen]
                    if not keep:
                        continue

                    if array is None:
                        hidden_size = int(features.shape[1])
                        capacity = len(blobs) * (MAX_ROWS_PER_BATCH or int(features.shape[0]))
                        array = open_memmap(features_path, mode='w+', dtype=np.float32,
                                            shape=(capacity, hidden_size))
                    take = min(len(keep), capacity - write_cursor)
                    if take <= 0:
                        continue
                    keep, keep_horizons = keep[:take], keep_horizons[:take]
                    array[write_cursor:write_cursor + take] = (
                        features[torch.as_tensor(keep, dtype=torch.long)].numpy()
                    )

                    for offset, months in zip(keep, keep_horizons):
                        record = flatten_metadata(metadata_rows[offset])
                        record['dataset'] = dataset.name
                        record['sample_index'] = int(payload['sample_indices'][offset])
                        record['batch_file'] = local_path.name
                        record['time_horizon_months'] = np.nan if months is None else months
                        record['log10_time_horizon_months'] = (
                            np.nan if months is None else float(np.log10(months))
                        )
                        if INCLUDE_PROMPTS:
                            record['prompt'] = payload['prompts'][offset]
                        records.append(record)
                    write_cursor += take
                    del payload, features
                finally:
                    if DELETE_BATCHES:
                        local_path.unlink(missing_ok=True)
                    progress.update(1)
    finally:
        executor.shutdown(wait=True)
        progress.close()
        if DELETE_BATCHES:
            shutil.rmtree(download_dir, ignore_errors=True)
        if array is not None:
            array.flush()
        del array
        gc.collect()

    if write_cursor == 0:
        raise ValueError(f'{dataset.name}: no rows were staged.')
    if dropped:
        print(f'  dropped {dropped:,} row(s) without a time horizon.')

    trim_to_rows(features_path, write_cursor)
    metadata_df = pd.DataFrame(records).reset_index(drop=True)
    # `row` is the row's index into the staged array, so the CSV and the `.npy` can be rejoined.
    metadata_df.insert(0, 'row', np.arange(len(metadata_df)))
    print(f'  staged {write_cursor:,} rows x {hidden_size:,} features -> {features_path.name}')
    return features_path, write_cursor, metadata_df


def open_features(features_path):
    """Open a staged feature array read-only."""
    return np.load(features_path, mmap_mode='r')

## 6. Run the extraction

In [ ]:
staged = {}
for dataset in DATASETS:
    print(f'=== {dataset.name} (gs://{BUCKET_NAME}/{folder_prefix(dataset)}) ===')
    features_path, row_count, metadata_df = stage_dataset(dataset)
    metadata_path = DATA_DIR / f'{dataset.name}_layer{LAYER}_pos{POSITION_TAG}_metadata.csv'
    metadata_df.to_csv(metadata_path, index=False)
    print(f'  metadata -> {metadata_path.name}')
    staged[dataset.name] = {
        'features_path': str(features_path),
        'metadata_path': str(metadata_path),
        'row_count': row_count,
    }

total_rows = sum(entry['row_count'] for entry in staged.values())
print(f'Staged {total_rows:,} rows across {len(staged)} dataset(s).')

## 7. Manifest and a look at what landed

The manifest records where each dataset's array and metadata CSV live, along with the settings
they were extracted under, so a downstream notebook can pick them up without re-deriving the
naming scheme.

In [ ]:
manifest = {
    'bucket': BUCKET_NAME,
    'range_tag': RANGE_TAG,
    'layer': LAYER,
    'layer_component': LAYER_COMPONENT,
    'positions': POSITIONS,
    'require_horizon': REQUIRE_HORIZON,
    'batch_sample_size': BATCH_SAMPLE_SIZE,
    'max_rows_per_batch': MAX_ROWS_PER_BATCH,
    'datasets': staged,
}
manifest_path = DATA_DIR / f'layer{LAYER}_pos{POSITION_TAG}_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f'Manifest -> {manifest_path}')

for name, entry in staged.items():
    path = Path(entry['features_path'])
    features = open_features(path)
    print(f'{name}: {features.shape} {features.dtype}, {path.stat().st_size / 1e9:.2f} GB on disk')
    del features

## 8. Clean up the downloads

Batch files are already unlinked as they are read and each dataset's download directory is
removed when its folder is exhausted, so this is a backstop: it clears anything a crashed or
interrupted run left behind and reports what remains. Only the extracted `.npy` arrays, the
metadata CSVs and the manifest should survive.

In [ ]:
batches_root = DATA_DIR / 'batches'
leftover = sorted(batches_root.rglob('*.pt')) if batches_root.exists() else []
if leftover:
    leftover_gb = sum(path.stat().st_size for path in leftover) / 1e9
    print(f'Removing {len(leftover):,} leftover batch file(s), {leftover_gb:.2f} GB.')
shutil.rmtree(batches_root, ignore_errors=True)
print('Batch downloads removed:', not batches_root.exists())

kept = sorted(path for path in DATA_DIR.rglob('*') if path.is_file())
kept_gb = sum(path.stat().st_size for path in kept) / 1e9
print(f'\nKept {len(kept)} file(s), {kept_gb:.2f} GB in {DATA_DIR}:')
for path in kept:
    print(f'  {path.relative_to(DATA_DIR)}  {path.stat().st_size / 1e9:.2f} GB')
print(f'\nFree on this volume: {shutil.disk_usage(DATA_DIR).free / 1e9:.1f} GB')

In [ ]:
# Sanity check on one dataset: the array rows line up with the metadata rows.
name = next(iter(staged))
entry = staged[name]
features = open_features(Path(entry['features_path']))
metadata_df = pd.read_csv(entry['metadata_path'])
assert features.shape[0] == len(metadata_df) == entry['row_count']
print(f'{name}: {features.shape[0]:,} rows aligned.')
print('Row norms (first 5):', np.linalg.norm(features[:5], axis=1).round(2))
metadata_df.head()